In [ ]:
"""
Reverse-JAKET Pipeline — KG -> GNN -> LM -> GNN -> KG (ComplEx scorer)
=======================================================================

What changed vs. the DistMult version:
1. Scorer swapped: DistMult -> ComplEx.
   - Entity/relation embeddings now hold a real part AND an imaginary part.
   - Implementation choice: each embedding is stored as ONE vector of size
     `2 * kg_dim` (first half = real component, second half = imaginary
     component). Both GNN encoders (Phase 1 / Phase 2) and the LM bridge
     operate on this doubled vector exactly as before -- they don't need to
     know it's "complex", they just see a bigger real vector. Only the
     scorer (score()) is aware of the real/imag split, since that's the only
     place the ComplEx math actually happens.
   - ComplEx score: Re( sum_k h_k * r_k * conj(t_k) )
       = sum( h_re*r_re*t_re + h_re*r_im*t_im + h_im*r_re*t_im - h_im*r_im*t_re )
     This is antisymmetric whenever r_im != 0, so it can represent
     antisymmetric relations like `located_in` (Paris located_in France is
     true, France located_in Paris is not) -- something DistMult cannot do,
     since DistMult's score(h,r,t) == score(t,r,h) always.

2. All the knobs you'd want to sweep by hand now live in one CONFIG dict at
   the top of the file (kg_dim, total_epochs, warmup_epochs, ramp_epochs,
   lr, lr_after_ramp, plus the existing data/LM settings).

3. Section 10's W&B grid-sweep code is gone. In its place: a plain single
   run driven directly by CONFIG, with matplotlib plots for loss and MRR.
   (W&B needed an account + was built for large grid sweeps; for local
   iteration this is simpler and has no external dependency.)

Everything else -- the live Wikidata SPARQL data loading, the frozen
RoBERTa bridge, the KG-only vs KG->LM->KG comparison, the t-SNE embedding
visualization, and the multi-seed stability check -- is unchanged in
structure, just running on ComplEx embeddings now.
"""

import requests, time, random, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.manifold import TSNE
from transformers import AutoTokenizer, AutoModel

SEED = 0
random.seed(SEED); torch.manual_seed(SEED); np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)
if device.type == "cpu":
    print("WARNING: no GPU detected. With thousands of entities this will be slow on CPU -- "
          "use a GPU runtime if you have one available.")


# ======================================================================
# 1. CONFIG -- every knob you'd want to hand-sweep lives here
# ======================================================================
CONFIG = {
    # --- the knobs you're sweeping ---
    'kg_dim':          350,     # PER-COMPONENT size. Choose: 250, 500, 800.
                                 # Actual embedding vector is 2*kg_dim (real+imag halves).
    'total_epochs':    500,     # Choose: 500 or 1000.
    'warmup_epochs':   20,      # epochs 1..warmup: pure GNN, alpha=0, no LM stage yet
    'ramp_epochs':     40,      # epochs after warmup: alpha ramps 0 -> 1
    'lr':              0.01,
    'lr_after_ramp':   0.003,

    # --- data / LM settings (unchanged from the original notebook) ---
    'num_cities':             10000,
    'min_city_population':    1000,
    'wikidata_country_limit': 200,
    'lm_model_name':          'roberta-base',
    'freeze_lm':              True,
    'max_desc_len':           24,
    'lm_batch_size':          256,
    'neg_per_pos':            1,
    'eval_every':              25,
}
print(CONFIG)


In [ ]:
# ======================================================================
# 2. Data — pulled live from Wikidata (identical to the original notebook)
# ======================================================================
WIKIDATA_SPARQL_URL = "https://query.wikidata.org/sparql"
HEADERS = {"User-Agent": "ReverseJaketPipeline/0.4 (student research project; contact: example@example.com)"}


def sparql(query, retries=6):
    """Exponential backoff, since the public endpoint returns 429 (rate limited) and
    502/504 (temporarily overloaded) fairly often under load -- these are transient,
    not a broken query, so waiting longer between retries usually gets through."""
    for attempt in range(retries):
        try:
            resp = requests.get(WIKIDATA_SPARQL_URL, params={"query": query, "format": "json"},
                                 headers=HEADERS, timeout=120)
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 10 * (attempt + 1)))
                print(f"attempt {attempt+1}: rate limited (429), waiting {wait}s before retrying")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            return resp.json()["results"]["bindings"]
        except Exception as e:
            wait = 5 * (2 ** attempt)
            print(f"attempt {attempt+1} failed: {e} -- waiting {wait}s before retrying")
            time.sleep(wait)
    raise RuntimeError(
        "Could not reach the Wikidata Query Service after several retries. "
        "This is usually transient server-side load -- wait a minute and re-run this cell. "
        "If it keeps failing, try lowering CONFIG['num_cities'] or CONFIG['min_city_population'] "
        "to make the query itself cheaper for Wikidata to answer."
    )


def qid(uri):
    return uri.rsplit("/", 1)[-1]


COUNTRY_QUERY = f"""
SELECT ?country ?countryLabel ?countryDesc ?capital ?capitalLabel ?capitalDesc
       ?continent ?continentLabel ?continentDesc WHERE {{
  ?country wdt:P31 wd:Q3624078 .
  ?country wdt:P36 ?capital .
  ?country wdt:P30 ?continent .
  OPTIONAL {{ ?country  schema:description ?countryDesc.   FILTER(LANG(?countryDesc)="en") }}
  OPTIONAL {{ ?capital  schema:description ?capitalDesc.   FILTER(LANG(?capitalDesc)="en") }}
  OPTIONAL {{ ?continent schema:description ?continentDesc. FILTER(LANG(?continentDesc)="en") }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT {CONFIG['wikidata_country_limit']}
"""

CITY_QUERY = f"""
SELECT ?city ?cityLabel ?cityDesc ?country ?countryLabel ?countryDesc WHERE {{
  ?city wdt:P31 wd:Q515 .
  ?city wdt:P1082 ?population .
  FILTER(?population > {CONFIG['min_city_population']})
  ?city wdt:P17 ?country .
  OPTIONAL {{ ?city schema:description ?cityDesc. FILTER(LANG(?cityDesc)="en") }}
  OPTIONAL {{ ?country schema:description ?countryDesc. FILTER(LANG(?countryDesc)="en") }}
  SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
}}
LIMIT {CONFIG['num_cities']}
"""

CLASS_QUERY = """
SELECT ?item ?itemLabel ?itemDesc WHERE {
  VALUES ?item { wd:Q6256 wd:Q515 wd:Q5107 }
  OPTIONAL { ?item schema:description ?itemDesc. FILTER(LANG(?itemDesc)="en") }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

country_rows = sparql(COUNTRY_QUERY)
city_rows = sparql(CITY_QUERY)
class_rows = sparql(CLASS_QUERY)
print(f"fetched {len(country_rows)} country rows, {len(city_rows)} city rows, {len(class_rows)} class rows from Wikidata")

DESCRIPTIONS = {}
NODE_CATEGORY = {}
LABELS = {}
country_ids, capital_ids, continent_ids, city_ids = set(), set(), set(), set()
raw_country_triples = []
raw_city_triples = []

for row in country_rows:
    c_qid = qid(row["country"]["value"]); c_label = row["countryLabel"]["value"]
    cap_qid = qid(row["capital"]["value"]); cap_label = row["capitalLabel"]["value"]
    cont_qid = qid(row["continent"]["value"]); cont_label = row["continentLabel"]["value"]

    DESCRIPTIONS[c_qid] = row.get("countryDesc", {}).get("value", f"a country called {c_label}")
    DESCRIPTIONS[cap_qid] = row.get("capitalDesc", {}).get("value", f"a city called {cap_label}")
    DESCRIPTIONS[cont_qid] = row.get("continentDesc", {}).get("value", f"a continent called {cont_label}")
    LABELS[c_qid] = c_label; LABELS[cap_qid] = cap_label; LABELS[cont_qid] = cont_label

    NODE_CATEGORY[c_qid] = "country"; NODE_CATEGORY[cap_qid] = "city"; NODE_CATEGORY[cont_qid] = "continent"
    country_ids.add(c_qid); capital_ids.add(cap_qid); city_ids.add(cap_qid); continent_ids.add(cont_qid)
    raw_country_triples.append((c_qid, cap_qid, cont_qid))

for row in city_rows:
    city_qid = qid(row["city"]["value"]); city_label = row["cityLabel"]["value"]
    country_qid = qid(row["country"]["value"]); country_label = row["countryLabel"]["value"]

    if city_qid not in DESCRIPTIONS:
        DESCRIPTIONS[city_qid] = row.get("cityDesc", {}).get("value", f"a city called {city_label}")
        LABELS[city_qid] = city_label
        NODE_CATEGORY[city_qid] = "city"
    city_ids.add(city_qid)

    if country_qid not in DESCRIPTIONS:
        DESCRIPTIONS[country_qid] = row.get("countryDesc", {}).get("value", f"a country called {country_label}")
        LABELS[country_qid] = country_label
        NODE_CATEGORY[country_qid] = "country"
    country_ids.add(country_qid)

    raw_city_triples.append((city_qid, country_qid))

CLASS_ID = {}
for row in class_rows:
    q = qid(row["item"]["value"])
    label = row["itemLabel"]["value"]
    DESCRIPTIONS[q] = row.get("itemDesc", {}).get("value", f"the class of {label}")
    LABELS[q] = label
    NODE_CATEGORY[q] = "class"
CLASS_QIDS = {"Country": "Q6256", "City": "Q515", "Continent": "Q5107"}
for name, real_qid in CLASS_QIDS.items():
    CLASS_ID[name] = real_qid

ENTITIES = sorted(country_ids) + sorted(city_ids - capital_ids) + sorted(capital_ids) + \
           sorted(continent_ids) + list(CLASS_ID.values())
ENTITIES = list(dict.fromkeys(ENTITIES))
ENTITY2ID = {e: i for i, e in enumerate(ENTITIES)}

RELATIONS = ["has_capital", "in_continent", "located_in", "is_a"]
RELATION2ID = {r: i for i, r in enumerate(RELATIONS)}
NUM_ENTITIES = len(ENTITIES)
NUM_RELATIONS = len(RELATIONS)

TRIPLES = []
for c_qid, cap_qid, cont_qid in raw_country_triples:
    TRIPLES.append((ENTITY2ID[c_qid], RELATION2ID["has_capital"], ENTITY2ID[cap_qid]))
    TRIPLES.append((ENTITY2ID[c_qid], RELATION2ID["in_continent"], ENTITY2ID[cont_qid]))
    TRIPLES.append((ENTITY2ID[c_qid], RELATION2ID["is_a"], ENTITY2ID[CLASS_ID["Country"]]))
for city_qid, country_qid in raw_city_triples:
    TRIPLES.append((ENTITY2ID[city_qid], RELATION2ID["located_in"], ENTITY2ID[country_qid]))
for city_qid in city_ids:
    TRIPLES.append((ENTITY2ID[city_qid], RELATION2ID["is_a"], ENTITY2ID[CLASS_ID["City"]]))
for cont_qid in continent_ids:
    TRIPLES.append((ENTITY2ID[cont_qid], RELATION2ID["is_a"], ENTITY2ID[CLASS_ID["Continent"]]))

TRIPLES = sorted(set(TRIPLES))
random.shuffle(TRIPLES)
n = len(TRIPLES)
n_train = int(n * 0.8); n_val = int(n * 0.1)
TRAIN = TRIPLES[:n_train]
VAL = TRIPLES[n_train:n_train + n_val]
TEST = TRIPLES[n_train + n_val:]
print(f"entities={NUM_ENTITIES} relations={NUM_RELATIONS} triples={n} -> train/val/test = {len(TRAIN)}/{len(VAL)}/{len(TEST)}")

REL_EDGES = {r: [] for r in range(NUM_RELATIONS)}
for h, r, t in TRAIN:
    REL_EDGES[r].append((h, t))

print("\nsample descriptions (all real, pulled live from Wikidata):")
for q in list(country_ids)[:2] + list(city_ids)[:3] + list(CLASS_ID.values()):
    print(f"  {LABELS[q]:20s} -> {DESCRIPTIONS[q]}")




In [ ]:
# ======================================================================
# 3. Model — GNN encoders (mini R-GCN) + projections + frozen RoBERTa + ComplEx scorer
# ======================================================================
# Design note: entity_emb / relation_emb now have width 2*kg_dim. Columns
# [0:kg_dim] are the "real" component, columns [kg_dim:2*kg_dim] are the
# "imaginary" component. KGEncoder, rgcn_layer, and the LM bridge are all
# dimension-agnostic, so they need no changes -- they just see a wider
# vector. Only score() splits the vector back into real/imag to apply the
# actual ComplEx formula.

def rgcn_layer(x, W_self, W_rels, rel_edges, num_entities):
    dim = x.shape[1]
    out = x @ W_self
    for r, edges in rel_edges.items():
        if not edges:
            continue
        src = torch.tensor([e[0] for e in edges], dtype=torch.long, device=x.device)
        dst = torch.tensor([e[1] for e in edges], dtype=torch.long, device=x.device)
        msg = x[src] @ W_rels[r]
        agg = torch.zeros(num_entities, dim, device=x.device)
        deg = torch.zeros(num_entities, 1, device=x.device)
        agg.index_add_(0, dst, msg)
        deg.index_add_(0, dst, torch.ones(len(dst), 1, device=x.device))
        deg = deg.clamp(min=1)
        out = out + agg / deg
    return F.relu(out)


class KGEncoder(nn.Module):
    """The GNN. One relation-aware graph-conv layer -- used twice (Phase 1, Phase 2),
    each with its own independently-trained weights. Trained from scratch, no pretrained GNN.
    Operates on whatever width vector it's given -- here that's the 2*kg_dim
    real+imaginary ComplEx vector, but the layer itself has no notion of "complex"."""
    def __init__(self, dim, num_relations):
        super().__init__()
        self.W_self = nn.Parameter(torch.randn(dim, dim) * 0.1)
        self.W_rels = nn.ParameterList([nn.Parameter(torch.randn(dim, dim) * 0.1) for _ in range(num_relations)])

    def forward(self, x, rel_edges, num_entities):
        return rgcn_layer(x, self.W_self, list(self.W_rels), rel_edges, num_entities)


class ReverseJaketPipeline(nn.Module):
    def __init__(self, num_entities, num_relations, kg_dim, lm_hidden, lm, tokenizer, max_len,
                 use_lm_stage, lm_batch_size):
        super().__init__()
        self.kg_dim = kg_dim               # per-component size
        full_dim = 2 * kg_dim              # real+imag concatenated width

        self.entity_emb   = nn.Parameter(torch.randn(num_entities, full_dim) * 0.1)
        self.relation_emb = nn.Parameter(torch.randn(num_relations, full_dim) * 0.1)
        self.phase1 = KGEncoder(full_dim, num_relations)
        self.phase2 = KGEncoder(full_dim, num_relations)

        self.kg_to_lm = nn.Linear(full_dim, lm_hidden)
        nn.init.normal_(self.kg_to_lm.weight, std=0.02)
        nn.init.zeros_(self.kg_to_lm.bias)
        self.soft_prompt_norm = nn.LayerNorm(lm_hidden)

        self.lm_to_kg = nn.Linear(lm_hidden, full_dim)
        nn.init.normal_(self.lm_to_kg.weight, std=0.02)
        nn.init.zeros_(self.lm_to_kg.bias)

        self.lm = lm
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.use_lm_stage = use_lm_stage
        self.lm_batch_size = lm_batch_size
        self.word_embeddings = lm.get_input_embeddings()

    def lm_conditioned(self, kg_vecs, desc_input_ids, desc_attention_mask):
        """Soft-prompt injection, processed in chunks of `lm_batch_size` entities at a time.
        kg_vecs here is the full 2*kg_dim (real+imag) vector -- RoBERTa doesn't need to know
        that; it's just a wider soft-prompt token."""
        n = kg_vecs.shape[0]
        outs = []
        ctx = torch.no_grad() if not any(p.requires_grad for p in self.lm.parameters()) else torch.enable_grad()
        for start in range(0, n, self.lm_batch_size):
            end = min(start + self.lm_batch_size, n)
            soft_prompt = self.kg_to_lm(kg_vecs[start:end])
            soft_prompt = self.soft_prompt_norm(soft_prompt).unsqueeze(1)
            word_embeds = self.word_embeddings(desc_input_ids[start:end])
            inputs_embeds = torch.cat([soft_prompt, word_embeds], dim=1)
            prompt_mask = torch.ones(end - start, 1, device=desc_attention_mask.device)
            attn_mask = torch.cat([prompt_mask, desc_attention_mask[start:end]], dim=1)
            with ctx:
                out = self.lm(inputs_embeds=inputs_embeds, attention_mask=attn_mask)
            mask = attn_mask.unsqueeze(-1)
            pooled = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(min=1)
            outs.append(pooled)
        pooled_all = torch.cat(outs, dim=0)
        return self.lm_to_kg(pooled_all)

    def encode_all(self, rel_edges, num_entities, desc_input_ids, desc_attention_mask, alpha=1.0):
        """alpha in [0,1]: 0 = pure GNN (KG-only), 1 = full LM stage, in-between = blended."""
        h1 = self.phase1(self.entity_emb, rel_edges, num_entities)
        if self.use_lm_stage and alpha > 0:
            lm_feats = self.lm_conditioned(h1, desc_input_ids, desc_attention_mask)
            feats = (1 - alpha) * h1 + alpha * lm_feats
        else:
            feats = h1
        h2 = self.phase2(feats, rel_edges, num_entities)
        return h2

    def score(self, h, r, t):
        """ComplEx scorer: Re( sum_k h_k * r_k * conj(t_k) )
        = h_re*r_re*t_re + h_re*r_im*t_im + h_im*r_re*t_im - h_im*r_im*t_re
        This is antisymmetric under swapping h<->t whenever r_im != 0, which is what
        lets it represent antisymmetric relations like `located_in` -- DistMult's
        score(h,r,t) == score(t,r,h) always, so it couldn't tell direction apart."""
        d = self.kg_dim
        h_re, h_im = h[..., :d], h[..., d:]
        r_re, r_im = r[..., :d], r[..., d:]
        t_re, t_im = t[..., :d], t[..., d:]
        return (h_re * r_re * t_re + h_re * r_im * t_im
                + h_im * r_re * t_im - h_im * r_im * t_re).sum(dim=-1)




In [ ]:
# ======================================================================
# 4. Load RoBERTa (frozen, unchanged) and tokenize descriptions
# ======================================================================
def build_lm(cfg):
    tokenizer = AutoTokenizer.from_pretrained(cfg["lm_model_name"])
    lm = AutoModel.from_pretrained(cfg["lm_model_name"])
    if cfg["freeze_lm"]:
        for p in lm.parameters():
            p.requires_grad = False
        lm.eval()
    lm.to(device)
    return lm, tokenizer


lm, tokenizer = build_lm(CONFIG)
LM_HIDDEN = lm.config.hidden_size
print("LM hidden size:", LM_HIDDEN)

desc_texts = [DESCRIPTIONS[e] for e in ENTITIES]
enc = tokenizer(desc_texts, padding="max_length", truncation=True,
                max_length=CONFIG["max_desc_len"], return_tensors="pt")
DESC_INPUT_IDS = enc["input_ids"].to(device)
DESC_ATTN_MASK = enc["attention_mask"].to(device)
print(DESC_INPUT_IDS.shape)




In [ ]:
# ======================================================================
# 5. Training + filtered evaluation (MRR, Hits@1/3/10)
# ======================================================================
ALL_TRUE_TAILS = {}
for h, r, t in TRIPLES:
    ALL_TRUE_TAILS.setdefault((h, r), set()).add(t)


def sample_negatives(batch, num_entities):
    neg = []
    for h, r, t in batch:
        if random.random() < 0.5:
            neg.append((random.randrange(num_entities), r, t))
        else:
            neg.append((h, r, random.randrange(num_entities)))
    return neg


def get_alpha(epoch, cfg):
    if epoch <= cfg["warmup_epochs"]:
        return 0.0
    ramp_pos = epoch - cfg["warmup_epochs"]
    if ramp_pos >= cfg["ramp_epochs"]:
        return 1.0
    return ramp_pos / cfg["ramp_epochs"]


def evaluate(model, triples_eval, rel_edges, num_entities, alpha=1.0, by_relation=False):
    model.eval()
    with torch.no_grad():
        final_emb = model.encode_all(rel_edges, num_entities, DESC_INPUT_IDS, DESC_ATTN_MASK, alpha=alpha)
    ranks = []
    ranks_by_rel = {r: [] for r in range(NUM_RELATIONS)}
    for h, r, t in triples_eval:
        r_e = model.relation_emb[r]
        # broadcast h,r against every candidate tail entity using the same ComplEx score()
        scores = model.score(final_emb[h].unsqueeze(0), r_e.unsqueeze(0), final_emb)
        target = scores[t].item()
        filtered = scores.clone()
        for tt in ALL_TRUE_TAILS.get((h, r), set()):
            if tt != t:
                filtered[tt] = -1e9
        rank = (filtered > target).sum().item() + 1
        ranks.append(rank)
        ranks_by_rel[r].append(rank)
    ranks_t = torch.tensor(ranks, dtype=torch.float)
    metrics = {
        "MRR": (1.0 / ranks_t).mean().item(),
        "Hits@1": (ranks_t <= 1).float().mean().item(),
        "Hits@3": (ranks_t <= 3).float().mean().item(),
        "Hits@10": (ranks_t <= 10).float().mean().item(),
    }
    if by_relation:
        metrics["by_relation"] = {}
        for r, rr in ranks_by_rel.items():
            if not rr:
                continue
            rr = torch.tensor(rr, dtype=torch.float)
            metrics["by_relation"][RELATIONS[r]] = {
                "n": len(rr), "MRR": (1.0 / rr).mean().item(), "Hits@10": (rr <= 10).float().mean().item()
            }
    return metrics, final_emb


def run_experiment(use_lm_stage, cfg, log_fn=None):
    model = ReverseJaketPipeline(NUM_ENTITIES, NUM_RELATIONS, cfg["kg_dim"], LM_HIDDEN,
                                  lm, tokenizer, cfg["max_desc_len"], use_lm_stage,
                                  cfg["lm_batch_size"]).to(device)
    trainable = [p for n, p in model.named_parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable, lr=cfg["lr"])

    history = {"loss": [], "val": [], "alpha": []}
    lr_dropped = False
    best_val_mrr = -1.0
    best_state = None

    for epoch in range(1, cfg["total_epochs"] + 1):
        model.train()
        alpha = get_alpha(epoch, cfg) if use_lm_stage else 0.0
        history["alpha"].append(alpha)

        if use_lm_stage and alpha > 0 and not lr_dropped:
            for g in optimizer.param_groups:
                g["lr"] = cfg["lr_after_ramp"]
            lr_dropped = True

        optimizer.zero_grad()
        final_emb = model.encode_all(REL_EDGES, NUM_ENTITIES, DESC_INPUT_IDS, DESC_ATTN_MASK, alpha=alpha)

        pos = TRAIN
        neg = sample_negatives(pos, NUM_ENTITIES) * cfg["neg_per_pos"]
        pos_h = torch.tensor([p[0] for p in pos], device=device)
        pos_r = torch.tensor([p[1] for p in pos], device=device)
        pos_t = torch.tensor([p[2] for p in pos], device=device)
        neg_h = torch.tensor([p[0] for p in neg], device=device)
        neg_r = torch.tensor([p[1] for p in neg], device=device)
        neg_t = torch.tensor([p[2] for p in neg], device=device)

        pos_scores = model.score(final_emb[pos_h], model.relation_emb[pos_r], final_emb[pos_t])
        neg_scores = model.score(final_emb[neg_h], model.relation_emb[neg_r], final_emb[neg_t])
        scores = torch.cat([pos_scores, neg_scores])
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)])
        loss = F.binary_cross_entropy_with_logits(scores, labels)   # BCE on real-vs-corrupted ComplEx scores
        loss.backward()
        optimizer.step()
        history["loss"].append(loss.item())

        val_metrics = None
        if epoch % cfg["eval_every"] == 0 or epoch == cfg["total_epochs"]:
            val_metrics, _ = evaluate(model, VAL, REL_EDGES, NUM_ENTITIES, alpha=alpha)
            history["val"].append((epoch, val_metrics))
            tag = "KG->LM->KG" if use_lm_stage else "KG-only"
            print(f"[{tag}] epoch {epoch:4d} | alpha {alpha:.2f} | loss {loss.item():.4f} | "
                  f"val MRR {val_metrics['MRR']:.3f} Hits@10 {val_metrics['Hits@10']:.3f}")
            if val_metrics["MRR"] > best_val_mrr:
                best_val_mrr = val_metrics["MRR"]
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

        if log_fn is not None:
            log_fn(epoch, loss.item(), alpha, val_metrics)

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[{'KG->LM->KG' if use_lm_stage else 'KG-only'}] restored best checkpoint, val MRR={best_val_mrr:.3f}")

    final_alpha = 1.0 if use_lm_stage else 0.0
    test_metrics, final_emb = evaluate(model, TEST, REL_EDGES, NUM_ENTITIES, alpha=final_alpha, by_relation=True)
    print(f"[{'KG->LM->KG' if use_lm_stage else 'KG-only'}] TEST:", test_metrics)
    return model, history, test_metrics, final_emb





In [ ]:
# ======================================================================
# 6. Run the two configs (KG-only baseline vs. proposed KG->LM->KG)
# ======================================================================
"""
model_kg_only, hist_kg_only, test_kg_only, emb_kg_only = run_experiment(use_lm_stage=False, cfg=CONFIG)
print()
model_full, hist_full, test_full, emb_full = run_experiment(use_lm_stage=True, cfg=CONFIG)

print("\nPer-relation test breakdown:")
print("KG-only    :", test_kg_only["by_relation"])
print("KG->LM->KG :", test_full["by_relation"])
"""


In [ ]:
# ======================================================================
# 7. Compare training curves: loss, MRR, and Hits@1/3/10
# ======================================================================
"""
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0, 0].plot(hist_kg_only["loss"], label="KG-only baseline")
axes[0, 0].plot(hist_full["loss"], label="KG -> LM -> KG (proposed)")
axes[0, 0].set_title("Training loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].set_ylabel("BCE loss")
axes[0, 0].legend()


def series(hist, key):
    return zip(*[(e, m[key]) for e, m in hist["val"]])


ek, mrr_k = series(hist_kg_only, "MRR"); ef, mrr_f = series(hist_full, "MRR")
axes[0, 1].plot(ek, mrr_k, marker="o", label="KG-only baseline")
axes[0, 1].plot(ef, mrr_f, marker="o", label="KG -> LM -> KG (proposed)")
axes[0, 1].axvspan(CONFIG["warmup_epochs"], CONFIG["warmup_epochs"] + CONFIG["ramp_epochs"],
                    color="gray", alpha=0.15, label="LM ramping in")
axes[0, 1].set_title("Validation MRR"); axes[0, 1].set_xlabel("epoch"); axes[0, 1].set_ylabel("MRR")
axes[0, 1].legend()

ek1, h1_k = series(hist_kg_only, "Hits@1"); ef1, h1_f = series(hist_full, "Hits@1")
axes[1, 0].plot(ek1, h1_k, marker="o", label="KG-only baseline")
axes[1, 0].plot(ef1, h1_f, marker="o", label="KG -> LM -> KG (proposed)")
axes[1, 0].set_title("Validation Hits@1"); axes[1, 0].set_xlabel("epoch"); axes[1, 0].set_ylabel("Hits@1")
axes[1, 0].legend()

ek10, h10_k = series(hist_kg_only, "Hits@10"); ef10, h10_f = series(hist_full, "Hits@10")
axes[1, 1].plot(ek10, h10_k, marker="o", label="KG-only baseline")
axes[1, 1].plot(ef10, h10_f, marker="o", label="KG -> LM -> KG (proposed)")
axes[1, 1].set_title("Validation Hits@10"); axes[1, 1].set_xlabel("epoch"); axes[1, 1].set_ylabel("Hits@10")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

metrics_names = ["MRR", "Hits@1", "Hits@3", "Hits@10"]
kg_vals = [test_kg_only[m] for m in metrics_names]
full_vals = [test_full[m] for m in metrics_names]
x = np.arange(len(metrics_names)); w = 0.35
plt.figure(figsize=(7, 4.5))
plt.bar(x - w / 2, kg_vals, w, label="KG-only baseline")
plt.bar(x + w / 2, full_vals, w, label="KG -> LM -> KG (proposed)")
plt.xticks(x, metrics_names)
plt.ylabel("score"); plt.title("Final TEST metrics (ComplEx scorer)")
plt.legend()
plt.tight_layout()
plt.savefig("test_metrics_bar.png", dpi=150)
plt.show()


"""

In [ ]:
# ======================================================================
# 8. Visualize how the embedding space changes across stages (t-SNE)
# ======================================================================
"""
CAT_COLORS = {"country": "#8ecae6", "city": "#219ebc", "continent": "#90be6d", "class": "#ffb703"}


def get_stage_embeddings(model, rel_edges, num_entities):
    model.eval()
    with torch.no_grad():
        raw = model.entity_emb.detach().cpu().numpy()
        h1 = model.phase1(model.entity_emb, rel_edges, num_entities).detach().cpu().numpy()
        final = model.encode_all(rel_edges, num_entities, DESC_INPUT_IDS, DESC_ATTN_MASK, alpha=1.0).detach().cpu().numpy()
    return raw, h1, final


raw_emb, phase1_emb, final_emb_np = get_stage_embeddings(model_full, REL_EDGES, NUM_ENTITIES)
colors = [CAT_COLORS[NODE_CATEGORY[e]] for e in ENTITIES]
label_these = set(sorted(continent_ids)) | set(CLASS_ID.values()) | set(sorted(country_ids)[:8])

fig, axes = plt.subplots(1, 3, figsize=(16, 5.5))
for ax, emb, title in zip(axes, [raw_emb, phase1_emb, final_emb_np],
                          ["Random init (re+im)", "After KG Phase 1", "After full KG->LM->KG"]):
    proj = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca").fit_transform(emb)
    ax.scatter(proj[:, 0], proj[:, 1], c=colors, s=15, alpha=0.6)
    for i, e in enumerate(ENTITIES):
        if e in label_these:
            ax.annotate(LABELS.get(e, e), (proj[i, 0], proj[i, 1]), fontsize=7)
    ax.set_title(title)
handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=c, markersize=10, label=k)
           for k, c in CAT_COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=4)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig("tsne_stages.png", dpi=150)
plt.show()

"""


In [ ]:
# ======================================================================
# 9. Multi-seed check (is the improvement real, or one lucky run?)
# ======================================================================

def run_multi_seed(use_lm_stage, cfg, seeds=(0, 1, 2)):
    all_metrics = []
    for s in seeds:
        random.seed(s); torch.manual_seed(s); np.random.seed(s)
        _, _, test_metrics, _ = run_experiment(use_lm_stage, cfg)
        all_metrics.append(test_metrics)
    out = {}
    for m in ["MRR", "Hits@1", "Hits@3", "Hits@10"]:
        vals = np.array([tm[m] for tm in all_metrics])
        out[m] = (vals.mean(), vals.std())
    return out


# NOTE: reruns training len(seeds) times per config -- reduce total_epochs first if slow.
quick_cfg = dict(CONFIG); quick_cfg["total_epochs"] = 60
kg_only_stats = run_multi_seed(False, quick_cfg, seeds=(0, 1, 2))
full_stats = run_multi_seed(True, quick_cfg, seeds=(0, 1, 2))

print("KG-only    (mean +/- std):", {k: f"{v[0]:.3f} +/- {v[1]:.3f}" for k, v in kg_only_stats.items()})
print("KG->LM->KG (mean +/- std):", {k: f"{v[0]:.3f} +/- {v[1]:.3f}" for k, v in full_stats.items()})



In [ ]:
# ======================================================================
# 10. Clean single local run (replaces the old W&B grid sweep)
# ======================================================================
# The old Section 10 launched a 5(kg_dim) x 2(lr) x 2(use_lm_stage) = 20-run W&B grid.
# That's genuinely useful for a full sweep, but heavy for quick local iteration and
# requires a W&B account. This block instead just runs ONE config -- whatever is
# currently set in CONFIG at the top of the file -- and plots loss/MRR locally with
# matplotlib, no external service needed.
#
# To sweep by hand: edit CONFIG['kg_dim'] (250/750/1000... note kg_dim here is the
# PER-COMPONENT size, so the real ComplEx vector is 2x that) and/or
# CONFIG['total_epochs'], then re-run this block.
#
# Reminder (same bottleneck as before): every KG vector passing through the LM bridge
# is projected down to RoBERTa-base's fixed hidden size (768) and back. Since our vector
# is now 2*kg_dim wide, that projection bottleneck kicks in past kg_dim=384 (768/2),
# not 768 -- worth flagging in the sweep write-up.

print("\n=== Section 10: single clean local run using current CONFIG ===")
_, hist_single, test_single, _ = run_experiment(use_lm_stage=True, cfg=CONFIG)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(hist_single["loss"])
axes[0].set_title(f"Loss (kg_dim={CONFIG['kg_dim']}, epochs={CONFIG['total_epochs']})")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("BCE loss")

ep, mrr = series(hist_single, "MRR")
axes[1].plot(ep, mrr, marker="o")
axes[1].set_title("Validation MRR")
axes[1].set_xlabel("epoch"); axes[1].set_ylabel("MRR")

plt.tight_layout()
plt.savefig("single_run_curves.png", dpi=150)
plt.show()

print("Single-run TEST metrics:", {k: v for k, v in test_single.items() if k != "by_relation"})
print("Per-relation:", test_single["by_relation"])
